# KOS Pipeline Demo
### Automated Knowledge Organisation for the National Library Board

**The problem:** KOS librarians at NLB maintain authoritative entity records in TTE (Thesaurus and Taxonomy Editor) — people, organisations, places, buildings, awards, events. They currently monitor news manually, copy-paste names into TTE to check for existing records, and update fields by hand. 2–3 hours per week per person.

**The solution:** A semi-automated pipeline that reads Singapore news articles and proposes structured, evidence-backed updates to TTE for human review and approval. Librarians stop doing surveillance work. They keep doing judgment work.

**One line:** In the past you had to read news because it was part of your job. Now you read it because you want to.

---

| Step | What | LLM? |
|---|---|---|
| 1 | Build BM25 index from TTE entity database | No |
| 2 | Link article entities to TTE records via BM25 | No |
| 3 | Extract specific TTE field values from article text | Optional |
| 4 | Assemble structured update proposals | No |
| 5 | Human review — approve / reject / no KB update / flag | No |
| 6 | Export approved changes as CSV for TTE import | No |

**No TTE writes without human approval. Every proposal is traceable to an exact article and evidence sentence.**

---
*Steps 1 and 2 have been pre-run. This notebook runs steps 4–6 live and shows the output at each stage.*

In [2]:
import sys
sys.path.insert(0, '..')

import json
import os
import pickle
import subprocess
import pandas as pd

import config
import utils

print('Setup complete.')

Setup complete.


## Step 1 — The data we start with

2,970 CNA articles were scraped and pre-processed using two LLM calls (already done):

- **Call 1 — classifier:** Given the article *title only*, classify the event type from a 37-type taxonomy, or return "not relevant". Cheap gate — filters ~85% of articles before spending on full text.
- **NER:** Given the full article text, extract named entities with labels (PERSON, ORGANISATION, FACILITY, etc.) and confidence scores.

Results are stored in `cna_articles.csv`. The pipeline picks up from entity linking.

In [3]:
articles = pd.read_csv('../data/cna_articles.csv', dtype=str).fillna('')

relevant = articles[
    (articles['Event_Type'] != 'not relevant') &
    (articles['Singapore_Context'].str.lower() == 'true')
]

print(f'Total articles scraped : {len(articles):,}')
print(f'Relevant articles      : {len(relevant):,}')
print(f'Relevance rate         : {len(relevant)/len(articles)*100:.1f}%')
print()
print('Top event types:')
print(relevant['Event_Type'].value_counts().head(10).to_string())

Total articles scraped : 2,970
Relevant articles      : 417
Relevance rate         : 14.0%

Top event types:
Event_Type
programme launched                        84
law enacted or amended                    42
building opened                           38
person received award                     35
person appointed or elected               33
building closed or ceased operations      31
organisation dissolved or closed          26
person died                               17
organisation restructured or rebranded    14
person resigned or retired                14


In [4]:
# Look at a sample article with its NER output
sample = relevant[relevant['Event_Type'] == 'building closed or ceased operations'].iloc[0]
print(f"Title      : {sample['Title']}")
print(f"Event type : {sample['Event_Type']}")
print(f"Reasoning  : {sample['Reasoning']}")
print()

entities = json.loads(sample['extracted_entities'])
print('Extracted entities:')
for e in entities:
    print(f"  [{e['entity_label']}] {e['entity_name']} (confidence: {e['confidence']})")

Title      : Don Don Donki to close Pasir Ris Downtown East store on Mar 15
Event type : building closed or ceased operations
Reasoning  : The Don Don Donki store at Pasir Ris Downtown East is permanently closing on March 15, which represents a specific named building ceasing operations.

Extracted entities:
  [ORGANISATION] Don Don Donki (confidence: high)
  [FACILITY] Downtown East (confidence: high)
  [LOCATION] Pasir Ris (confidence: high)
  [ORGANISATION] Facebook (confidence: high)
  [FACILITY] Tampines 1 (confidence: high)
  [FACILITY] Jewel Changi Airport (confidence: high)
  [COUNTRY] Singapore (confidence: high)
  [FACILITY] Orchard Central (confidence: high)
  [ORGANISATION] Don Quijote (confidence: high)


In [5]:
# How the classifier categorised different event types
print('Event classification examples:\n')

sample_types = [
    'person died', 'person appointed or elected',
    'building opened', 'organisation dissolved or closed'
]
for etype in sample_types:
    rows = relevant[relevant['Event_Type'] == etype]
    if rows.empty:
        continue
    row = rows.iloc[0]
    print(f"  Event type : {row['Event_Type']}")
    print(f"  Title      : {row['Title'][:75]}")
    print(f"  Reasoning  : {row['Reasoning'][:115]}...")
    print()

Event classification examples:

  Event type : person died
  Title      : Driver killed by metal piece shot from defective Takata airbag likely did n
  Reasoning  : A driver was killed by a metal piece from a defective Takata airbag, which constitutes a death by accident....

  Event type : person appointed or elected
  Title      : Urban Redevelopment Authority to get new CEO; next chief planner appointed
  Reasoning  : A new CEO and chief planner were appointed to specific roles at the Urban Redevelopment Authority, representing con...

  Event type : building opened
  Title      : New Inclusive Justice Law Centre to help those with special needs, mental h
  Reasoning  : The Inclusive Justice Law Centre is a new specific named organisation that has been established and is now operatio...

  Event type : organisation dissolved or closed
  Title      : Australian tea brand T2 Tea closing all outlets in Singapore
  Reasoning  : T2 Tea is closing all its outlets in Singapore, which means

## Step 2 — BM25 entity search index

TTE contains ~55,000 entities across 8 vocabularies with ~84,000 alias surfaces — authorised names, display names, UF variants ("Used For" — acronyms, former names, alternate spellings), and cross-language forms (English ↔ Chinese ↔ Malay ↔ Tamil).

We build a BM25 full-text search index over all these surfaces. When an article mentions "URA", the index returns "Urban Redevelopment Authority (Singapore)". When it says "Liu Thai Ker", it finds the correct person record across 22,000+ people.

**Why BM25 not vector search:** Entity queries from news are proper noun surface strings. "Lee Kuan Yew" matching "Lee Kuan Yew" is a lexical problem, not a semantic one. BM25 is faster, simpler, and more accurate for this specific retrieval pattern.

In [6]:
with open('../outputs/bm25_index.pkl', 'rb') as f:
    index = pickle.load(f)

print('BM25 Index loaded.')
print()
for vocab, vocab_index in index.items():
    n_entities = len(set(m['uid'] for m in vocab_index['metadata']))
    n_surfaces = len(vocab_index['surfaces'])
    print(f'  {vocab:<15} : {n_entities:,} entities, {n_surfaces:,} alias surfaces')

BM25 Index loaded.

  PERSON          : 22,258 entities, 40,300 alias surfaces
  ORGANISATION    : 11,775 entities, 20,712 alias surfaces
  FACILITY        : 3,418 entities, 6,824 alias surfaces
  LOCATION        : 8,998 entities, 14,429 alias surfaces
  EVENT           : 882 entities, 1,697 alias surfaces
  LEGAL_ACT       : 186 entities, 189 alias surfaces
  PROGRAMME       : 138 entities, 156 alias surfaces
  AWARD           : 368 entities, 510 alias surfaces


In [8]:
# Exact name match
query = 'Liu Thai Ker'
vocab = 'PERSON'

results = utils.search_index(index, query, vocab, top_k=5)

print(f"Query: '{query}' in {vocab}")
print()
if results:
    for r in results:
        print(f"  Score: {r['score']:.4f}  |  UID: {r['uid']}  |  {r['authorised_name']}")
        print(f"  Matched surface: '{r['matched_surface']}'  |  Record richness: {r['record_richness']} fields")
        print()
else:
    print('  No matches found.')

Query: 'Liu Thai Ker' in PERSON

  Score: 20.5784  |  UID: 18540823  |  Liu, Thai Ker
  Matched surface: 'Liu Thai Ker'  |  Record richness: 26 fields

  Score: 20.5784  |  UID: 18540823  |  Liu, Thai Ker
  Matched surface: 'Liu, Thai Ker'  |  Record richness: 26 fields

  Score: 20.5784  |  UID: 18552357  |  Liu Thai Ker
  Matched surface: 'Liu Thai Ker'  |  Record richness: 2 fields

  Score: 9.1701  |  UID: 18544105  |  Thai, Ha
  Matched surface: 'Thai Ha'  |  Record richness: 10 fields

  Score: 9.1701  |  UID: 18544105  |  Thai, Ha
  Matched surface: 'Thai, Ha'  |  Record richness: 10 fields



In [9]:
# Alias / acronym match — 'URA' resolves to full authorised name via UF index
query = 'URA'
vocab = 'ORGANISATION'

results = utils.search_index(index, query, vocab, top_k=3)
print(f"Query: '{query}' in {vocab}")
for r in results:
    print(f"  Score: {r['score']:.4f}  |  UID: {r['uid']}  |  {r['authorised_name']}")

Query: 'URA' in ORGANISATION
  Score: 13.6916  |  UID: 18487958  |  Urban Redevelopment Authority (Singapore)
  Score: 13.6916  |  UID: 18526475  |  URA


In [10]:
# Score thresholding — what separates high match from low match
for query, vocab in [('National Gallery Singapore', 'FACILITY'), ('Substation', 'FACILITY')]:
    results = utils.search_index(index, query, vocab, top_k=3)
    print(f"Query: '{query}' → {vocab}")
    for r in results:
        tag = '  ← HIGH MATCH ✓' if r['score'] >= config.HIGH_MATCH_THRESHOLD else ''
        print(f"  {r['score']:>8.4f}  {r['authorised_name']}{tag}")
    print()

print(f"HIGH_MATCH_THRESHOLD = {config.HIGH_MATCH_THRESHOLD}")
print("Scores above this → proceed to proposals automatically.")
print("Scores below this → surfaced for human review as low-confidence candidates.")

Query: 'National Gallery Singapore' → FACILITY
   15.7961  National Gallery Singapore (Place)  ← HIGH MATCH ✓
   14.0392  国家美术馆  ← HIGH MATCH ✓
   14.0392  National Gallery Singapore (Place)  ← HIGH MATCH ✓

Query: 'Substation' → FACILITY
    9.8329  The Substation (Arts centre)  ← HIGH MATCH ✓
    7.6457  The Substation (Arts centre)  ← HIGH MATCH ✓
    7.6457  电力站 (艺术中心)  ← HIGH MATCH ✓

HIGH_MATCH_THRESHOLD = 1.0
Scores above this → proceed to proposals automatically.
Scores below this → surfaced for human review as low-confidence candidates.


## Step 3 — Entity linking

For each relevant article, we identify the primary entity based on event type (e.g. `person died` → search PERSON vocabulary) and run a BM25 query against the corresponding TTE index.

Results fall into four categories:
- **high_match** — score ≥ threshold → entity confidently found in TTE → proceed to proposals
- **low_match** — score below threshold → candidate found but uncertain → surface for human review
- **no_match** — nothing found → potential new entity candidate → flagged
- **no_entity_found** — NER didn't extract an entity of the expected type for this article

In [11]:
linked = pd.read_csv('../outputs/linked_entities.csv', dtype=str).fillna('')

print(f'Total linked rows : {len(linked):,}')
print()
print('Match type distribution:')
print(linked['match_type'].value_counts().to_string())

Total linked rows : 1,915

Match type distribution:
match_type
high_match         1376
no_match            463
no_entity_found      62
low_match            14


In [12]:
# Sample high-confidence matches
high = linked[linked['match_type'] == 'high_match'].head(5)

print('Sample high-confidence matches:')
print()
for _, row in high.iterrows():
    print(f"  Article    : {row['article_title'][:60]}")
    print(f"  Event      : {row['event_type']}")
    print(f"  Extracted  : {row['entity_name']}")
    print(f"  → TTE match: {row['tte_authorised_name']}  (UID: {row['tte_uid']}, score: {row['match_score']})")
    print()

Sample high-confidence matches:

  Article    : DBS named world’s best private bank at global awards
  Event      : organisation received award
  Extracted  : DBS
  → TTE match: DBS Bank  (UID: 18545074, score: 12.024)

  Article    : DBS named world’s best private bank at global awards
  Event      : organisation received award
  Extracted  : DBS Digital Exchange
  → TTE match: DBS Bank  (UID: 18545074, score: 12.024)

  Article    : DBS named world’s best private bank at global awards
  Event      : organisation received award
  Extracted  : DBS Trustee
  → TTE match: DBS Bank  (UID: 18545074, score: 12.024)

  Article    : DBS named world’s best private bank at global awards
  Event      : organisation received award
  Extracted  : DBS Private Bank
  → TTE match: DBS Bank  (UID: 18545074, score: 17.7518)

  Article    : Urban Redevelopment Authority to get new CEO; next chief pla
  Event      : person appointed or elected
  Extracted  : Lim Eng Hwee
  → TTE match: Eng, Hwee Chu  (UI

In [13]:
# Low matches — found a candidate but score below threshold
low = linked[linked['match_type'] == 'low_match'].head(3)
if not low.empty:
    print('Low-confidence matches (surfaced for human review):\n')
    for _, row in low.iterrows():
        print(f"  Query   : {row['entity_name']}")
        print(f"  Best    : {row['tte_authorised_name']}  (score: {row['match_score']})")
        print(f"  Article : {row['article_title'][:60]}")
        print()

# No matches — entities TTE doesn't know about yet
no_match = linked[linked['match_type'] == 'no_match'].head(3)
if not no_match.empty:
    print('No TTE match found (potential new entity candidates):\n')
    for _, row in no_match.iterrows():
        print(f"  [{row['entity_type']}] {row['entity_name']}")
        print(f"  Article : {row['article_title'][:60]}")
        print()

Low-confidence matches (surfaced for human review):

  Query   : Ethnic Fusion Dance Programme
  Best    : Plant-A-Tree Programme  (score: 0.8991)
  Article : Singapore to introduce new S$20 million grant to support dev

  Query   : Bus Connectivity Enhancement Programme
  Best    : Plant-A-Tree Programme  (score: 0.8991)
  Article : Self-driving vehicles in Punggol to open to public from Apr 

  Query   : TechSkills Accelerator Programme
  Best    : Plant-A-Tree Programme  (score: 0.8991)
  Article : New training academy to equip more than 150,000 public offic

No TTE match found (potential new entity candidates):

  [ORGANISATION] Euromoney
  Article : DBS named world’s best private bank at global awards

  [ORGANISATION] DDEx
  Article : DBS named world’s best private bank at global awards

  [PROGRAMME] BCRS
  Article : New search function launched for Singapore residents to find



## Step 4 — Fact extraction *(optional LLM step)*

For high-confidence matches, a second LLM call can extract specific TTE field values directly from the article text — with the exact evidence sentence as provenance.

The field to extract depends on the event type. For example:
- `person died` → Death Year, Death Month, Death Day
- `person appointed or elected` → Affiliations, Occupation, Description
- `building renamed` → Name

**This step is optional for the POC.** Without it, proposals still flow through the review queue — librarians see the article URL, entity match, and event type, and fill in TTE fields themselves. Enable it in `config.py` when ready by setting `OPENROUTER_API_KEY` or `ANTHROPIC_API_KEY`.

In [10]:
# Field mapping: event type → TTE fields to extract
print('Field mapping (event type → TTE fields to extract):')
print()
for event_type, fields in list(config.FIELD_MAPPING.items())[:8]:
    print(f"  {event_type}")
    print(f"    → {', '.join(fields)}")

Field mapping (event type → TTE fields to extract):

  person died
    → Death Year (yyyy), Death Month (mm), Death Day (dd)
  person appointed or elected
    → Affiliations(groupName), Occupation, Active Years, Description
  person resigned or retired
    → Active Years, Affiliations(groupName), Occupation, Description
  person changed affiliation or party
    → Affiliations(groupName)
  person received award
    → Awards
  person received honour or title
    → Achievements
  person got married
    → Spouse
  person renamed
    → Name


In [14]:
if os.path.exists('../outputs/extracted_facts.csv'):
    facts = pd.read_csv('../outputs/extracted_facts.csv', dtype=str).fillna('')
    extracted = facts[facts['fact_status'] == 'extracted']
    print(f'Fact extraction has been run.')
    print(f'  Total processed        : {len(facts):,}')
    print(f'  Successfully extracted : {len(extracted):,}')
    print()
    for _, row in extracted.head(3).iterrows():
        fields = json.loads(row.get('extracted_fields', '{}'))
        non_null = {k: v['value'] for k, v in fields.items() if v and v.get('value')}
        if non_null:
            print(f"  {row['article_title'][:60]}")
            print(f"  Entity    : {row['entity_name']} → {row['tte_authorised_name']}")
            print(f"  Extracted : {non_null}")
            print()
else:
    print('Fact extraction not run (optional — skipped for now).')
    print()
    print('Proposals will be generated without pre-filled field values.')
    print('Librarians review the article directly and enter TTE fields manually.')
    print()
    print('To enable: set OPENROUTER_API_KEY (or ANTHROPIC_API_KEY) in config.py and run:')
    print('  python 03_extract_facts.py')

Fact extraction not run (optional — skipped for now).

Proposals will be generated without pre-filled field values.
Librarians review the article directly and enter TTE fields manually.

To enable: set OPENROUTER_API_KEY (or ANTHROPIC_API_KEY) in config.py and run:
  python 03_extract_facts.py


## Step 5 — Build proposals

All linked entities are assembled into structured update proposals and sorted by review priority:

- **HIGH** — high entity match + field values extracted by LLM + entity confirmed
- **MEDIUM** — high entity match (no fact extraction), or low-confidence match requiring disambiguation
- **FLAG** — no TTE match found — potential new entity candidate

No API calls. Pure code.

In [15]:
result = subprocess.run(
    ['python', '04_build_proposals.py'],
    capture_output=True, text=True, cwd='..'
)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr[:500])


  Step 4: Building Update Proposals

→ Loading linked entities from outputs/linked_entities.csv
  No extracted_facts.csv found — skipping fact extraction step.
  All rows will be proposed without pre-filled field values.
  Linked entities: 1,915 rows (1,915 without extracted facts)

→ Assembling proposals

────────────────────────────────────────
Proposals built.
  Total proposals : 1,915
  HIGH            : 0
  MEDIUM          : 1,452
  FLAG            : 463

  Saved to: outputs/proposals.json

Next step: python 05_review_queue.py



In [16]:
with open('../outputs/proposals.json') as f:
    proposals = json.load(f)

priority_counts = {}
for p in proposals:
    priority_counts[p['priority']] = priority_counts.get(p['priority'], 0) + 1

print(f'Total proposals : {len(proposals):,}')
for priority in ['HIGH', 'MEDIUM', 'FLAG']:
    print(f'  {priority:<8} : {priority_counts.get(priority, 0):,}')

Total proposals : 1,915
  HIGH     : 0
  MEDIUM   : 1,452
  FLAG     : 463


In [17]:
# Show a sample proposal — same format as the terminal review queue
sample = next(
    (p for p in proposals if p['priority'] == 'MEDIUM' and p['entity']['tte_uid']),
    None
)

if sample:
    e = sample['entity']
    print('=' * 68)
    print(f"  PROPOSAL #{sample['id']}   🟡 {sample['priority']} PRIORITY")
    print('=' * 68)
    print(f"\n  ARTICLE")
    print(f"  {sample['article_title']}")
    print(f"  {sample['article_url']}")
    print(f"\n  EVENT TYPE")
    print(f"  {sample['event_type']}")
    print(f"\n  ENTITY MATCH")
    print(f"  Extracted name   : {e['name']}")
    print(f"  Authorised form  : {e['authorised_name']}")
    print(f"  TTE UID          : {e['tte_uid']}")
    print(f"  Vocabulary       : {e['vocabulary']}")
    print(f"  Match score      : {e['match_score']:.3f}  ({e['match_type']})")
    print(f"  Record richness  : {e['record_richness']} existing TTE fields")
    if sample.get('note'):
        print(f"\n  NOTE: {sample['note']}")
    print('\n' + '─' * 68)
    print('  [a] Approve    [r] Reject    [n] No KB update    [f] Flag    [s] Skip')
    print('─' * 68)

  PROPOSAL #1   🟡 MEDIUM PRIORITY

  ARTICLE
  DBS named world’s best private bank at global awards
  https://www.channelnewsasia.com/singapore/dbs-best-safest-private-bank-global-awards-euromoney-6006691

  EVENT TYPE
  organisation received award

  ENTITY MATCH
  Extracted name   : DBS
  Authorised form  : DBS Bank
  TTE UID          : 18545074
  Vocabulary       : ORGANISATION
  Match score      : 12.024  (high_match)
  Record richness  : 17 existing TTE fields

  NOTE: Good entity match — no fact extraction run, review article directly

────────────────────────────────────────────────────────────────────
  [a] Approve    [r] Reject    [n] No KB update    [f] Flag    [s] Skip
────────────────────────────────────────────────────────────────────


## Step 6 — Human review

The review queue (`05_review_queue.py`) is a terminal interface. Run it from the project root:

```
python 05_review_queue.py
```

For each proposal the reviewer sees the article, matched TTE entity, event type, any pre-extracted field values with evidence, and near-match candidates for disambiguation.

**Controls:** `a` approve · `r` reject · `n` no KB update needed · `f` flag for discussion · `s` skip · `q` quit and save

Progress is saved after every decision. Resume at any time by rerunning the script.

---
*Below: simulated review decisions to show the flow.*

In [21]:
# Simulate reviewer decisions
approved_proposals = [p for p in proposals if p['priority'] == 'MEDIUM' and p['entity']['tte_uid']][:3]
flagged_proposals  = [p for p in proposals if p['priority'] == 'FLAG'][:1]

mock_decisions = []

print('Simulated reviewer decisions:\n')
for p in approved_proposals:
    mock_decisions.append({
        'proposal_id': str(p['id']),
        'decision':    'approved',
        'reason':      '',
        'reviewer':    'demo',
        'timestamp':   utils.now_str(),
        'article_url': p.get('article_url', ''),
        'entity_name': p['entity'].get('name', ''),
        'tte_uid':     p['entity'].get('tte_uid', ''),
        'event_type':  p.get('event_type', ''),
    })
    print(f"APPROVED  {p['entity']['name'][:35]:<35}  {p['event_type'][:35]}")

for p in flagged_proposals:
    mock_decisions.append({
        'proposal_id': str(p['id']),
        'decision':    'flagged',
        'reason':      'No TTE record — potential new entity candidate',
        'reviewer':    'demo',
        'timestamp':   utils.now_str(),
        'article_url': p.get('article_url', ''),
        'entity_name': p['entity'].get('name', ''),
        'tte_uid':     '',
        'event_type':  p.get('event_type', ''),
    })
    print(f" FLAGGED {p['entity']['name'][:35]:<35}  {p['event_type'][:35]}")

with open('../outputs/decisions.json', 'w') as f:
    json.dump(mock_decisions, f, indent=2, ensure_ascii=False)
print(f"\n  {len(mock_decisions)} decisions saved → outputs/decisions.json")

Simulated reviewer decisions:

APPROVED  DBS                                  organisation received award
APPROVED  DBS Digital Exchange                 organisation received award
APPROVED  DBS Trustee                          organisation received award
 FLAGGED Euromoney                            organisation received award

  4 decisions saved → outputs/decisions.json


## Step 7 — Export

Approved decisions are exported as two files:

- **`approved_changes.csv`** — one row per proposed field update, ready for manual TTE import
- **`decision_log.csv`** — every decision with full context — audit trail and future LLM training data

In [22]:
result = subprocess.run(
    ['python', '06_export.py'],
    capture_output=True, text=True, cwd='..'
)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr[:500])

# Show approved changes
if os.path.exists('../outputs/approved_changes.csv'):
    approved = pd.read_csv('../outputs/approved_changes.csv', dtype=str).fillna('')
    print(f'approved_changes.csv — {len(approved)} rows ready for TTE import\n')
    cols = ['tte_uid', 'tte_authorised_name', 'tte_vocabulary', 'event_type', 'entity_name', 'note']
    print(approved[cols].to_string(index=False))
    print()

# Show decision log
if os.path.exists('../outputs/decision_log.csv'):
    log = pd.read_csv('../outputs/decision_log.csv', dtype=str).fillna('')
    print(f'decision_log.csv — {len(log)} entries (audit trail + future training data)\n')
    cols = ['proposal_id', 'decision', 'reviewer', 'entity_name', 'event_type', 'tte_vocabulary', 'match_type']
    print(log[cols].to_string(index=False))


  Step 6: Exporting Approved Changes

→ Loading proposals and decisions
  Proposals loaded  : 1,915
  Decisions loaded  : 4

→ Building approved changes export

→ Building full decision log

────────────────────────────────────────
Export complete.

  Decision summary:
    approved             : 3
    flagged              : 1

  Approved changes   : 3 field updates
  Saved to           : outputs/approved_changes.csv

  Decision log       : 4 entries
  Saved to           : outputs/decision_log.csv

  The approved_changes.csv is ready for manual TTE import.
  The decision_log.csv is your audit trail and future training data.

approved_changes.csv — 3 rows ready for TTE import

 tte_uid tte_authorised_name tte_vocabulary                  event_type          entity_name                                                                                  note
18545074            DBS Bank   ORGANISATION organisation received award                  DBS Approved — no specific field changes extrac

---

## Pipeline summary

```
2,970 CNA articles scraped
    ↓  Call 1: LLM classifier on title — already done
417 relevant articles (14%)
    ↓  NER: entity extraction — already done
    ↓  Step 1: BM25 index built from TTE — code only, no LLM
    ↓  Step 2: Entity linking via BM25 — code only, no LLM
1,376 high-confidence TTE matches  |  463 no-match (new entity candidates)
    ↓  Step 3: LLM fact extraction — optional (skipped for now)
    ↓  Step 4: Build proposals — code only
Proposals sorted: HIGH → MEDIUM → FLAG
    ↓  Step 5: Human review — approve / reject / no KB update / flag
    ↓  Step 6: Export — code only
approved_changes.csv  →  manual TTE import
decision_log.csv      →  audit trail + future model training data
```

**Design principles:**
- LLMs used for data generation only — not final predictions
- All updates traceable to exact article text
- Precision over recall — false positives waste librarian time
- Human review is not a fallback — it is the point
- No TTE writes without human approval
- The decision log becomes labelled training data for future fine-tuning